# Natural Language Processing (NLP)
## Practical Notebook

**Topics covered:**
1. Bag of Words, Visual Vectors & NLP Pipeline
2. Web Scraping — `requests`/`BeautifulSoup` and `Selenium`
3. Sentiment Analysis — Document, Sentence & Aspect Level
4. Sentiment Approaches — Lexicon, Classical ML, Deep Learning
5. Retrieval-Augmented Generation (RAG)

In [ ]:
import sys, subprocess

# Install required packages into the active kernel's Python environment.
# --prefer-binary avoids source builds (required for faiss-cpu, torch on macOS).
# transformers<4.47 and sentence-transformers<4 required because torch max on
# Python 3.11 macOS is 2.2.2, and transformers>=4.47 requires torch>=2.4.
# numpy<2 required because torch 2.2.2 was compiled with numpy 1.x;
# torch.Tensor.numpy() raises RuntimeError with numpy>=2.
_pkgs = [
    "nltk", "scikit-learn", "matplotlib", "beautifulsoup4", "requests",
    "selenium", "webdriver-manager",
    "torch", "faiss-cpu",
    "numpy<2",                 # torch 2.2.2 incompatible with numpy>=2 (.numpy() breaks)
    "transformers<4.47",       # torch>=2.4 required by 4.47+; max for Py3.11 macOS is 2.2.2
    "sentence-transformers<4", # 4.x requires transformers>=5
    "vaderSentiment", "spacy", "gensim", "certifi", "pandas"
]
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", "--prefer-binary"] + _pkgs
)
print("All packages installed successfully.")


---
## 1. Bag of Words, Visual Vectors & NLP Pipeline

> **Source text**: Opening paragraph of *Moby Dick* (Melville, 1851) — public domain.

In [ ]:
import re
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from collections import Counter

# Opening of "Cien años de soledad" — Gabriel García Márquez (1967)
# Rich in repetition: "años", "soledad", "Macondo", "hielo", "coronel" — ideal for BoW
raw_text = (
    "Many years later, as he faced the firing squad, Colonel Aureliano Buendía "
    "was to remember that distant afternoon when his father took him to discover ice. "
    "At that time Macondo was a village of twenty adobe houses built on the bank of a "
    "river of clear water that ran along a bed of polished stones, which were white and "
    "enormous, like prehistoric eggs. The world was so recent that many things lacked "
    "names, and in order to point them out it was necessary to point. Every year during "
    "the month of March a family of ragged gypsies would set up their tents near the "
    "village, and with a great uproar of pipes and kettledrums they would display new "
    "inventions. First they brought the magnet. A heavy gypsy with an untamed beard and "
    "sparrow hands, who introduced himself as Melquíades, put on a bold public "
    "demonstration of what he himself called the eighth wonder of the learned alchemists "
    "of Macedonia. He went from house to house dragging two metal ingots and everybody "
    "was frightened to see that pots, pans, tongs, and braziers tumbled down from their "
    "places and beams creaked from the desperation of nails and screws trying to emerge, "
    "and even objects that had been lost for a long time appeared from where they had "
    "been searched for most and went dragging along in turbulent confusion behind "
    "Melquíades magical iron. Colonel Aureliano Buendía remembered the afternoon, "
    "remembered Macondo, remembered the ice, remembered many years of soledad and war."
)
print("RAW TEXT:\n", raw_text)


In [ ]:
import ssl
import certifi
import nltk
import nltk.internals  # Required on Python 3.14+ — fixes AttributeError in nltk.downloader

# macOS + pyenv: system SSL certs are not bundled with Python, patch with certifi
ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())

for _resource in ["stopwords", "wordnet", "omw-1.4", "punkt", "punkt_tab"]:
    nltk.download(_resource, quiet=True)

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

STOPWORDS = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


def nlp_pipeline(text: str) -> dict:
    """5-step NLP preprocessing pipeline."""
    # Step 1 – Tokenize
    tokens = word_tokenize(text)
    print(f"Step 1 – Tokens ({len(tokens)}):         {tokens[:10]} …")

    # Step 2 – Lowercase
    tokens_lower = [t.lower() for t in tokens]
    print(f"Step 2 – Lowercase: {tokens_lower[:10]} …")

    # Step 3 – Remove punctuation & stopwords
    tokens_clean = [t for t in tokens_lower if t.isalpha() and t not in STOPWORDS]
    print(f"Step 3 – After stopword removal: {tokens_clean[:10]} …")

    # Step 4 – Lemmatize
    tokens_lemma = [lemmatizer.lemmatize(t) for t in tokens_clean]
    print(f"Step 4 – Lemmatized: {tokens_lemma[:10]} …")

    # Step 5 – Build BoW frequency vector
    bow = Counter(tokens_lemma)
    print(f"Step 5 – BoW top-10: {bow.most_common(10)}")
    return {"tokens": tokens_lemma, "bow": bow}


result = nlp_pipeline(raw_text)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# ── BoW bar chart + sparse vector heatmap ────────────────────────────────────
bow   = result["bow"]
top_n = 15
words, counts = zip(*bow.most_common(top_n))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Bar chart — show top-N lemmatised tokens from the García Márquez paragraph
colors = ["#c0392b" if c >= 3 else "#2980b9" for c in counts[::-1]]
axes[0].barh(words[::-1], counts[::-1], color=colors)
axes[0].set_xlabel("Term Frequency")
axes[0].set_title(f"Top-{top_n} BoW — Cien años de soledad (García Márquez)")
for i, v in enumerate(counts[::-1]):
    axes[0].text(v + 0.05, i, str(v), va="center", fontsize=9)
red_patch  = mpatches.Patch(color="#c0392b", label="freq ≥ 3")
blue_patch = mpatches.Patch(color="#2980b9", label="freq < 3")
axes[0].legend(handles=[red_patch, blue_patch], fontsize=8)

# Sparse vector heatmap — two sentences drawn from the same text
vocab = [w for w, _ in bow.most_common(20)]
sent1 = "Colonel Aureliano Buendia remembered the distant afternoon when his father discovered ice"
sent2 = "Many years later Macondo was a village and the world lacked names for many things"

def to_bow_vec(sentence, vocab):
    tokens = [lemmatizer.lemmatize(t.lower()) for t in sentence.split() if t.isalpha()]
    c = Counter(tokens)
    return [c.get(w, 0) for w in vocab]

mat = np.array([to_bow_vec(sent1, vocab), to_bow_vec(sent2, vocab)], dtype=float)
im  = axes[1].imshow(mat, aspect="auto", cmap="YlOrRd")
axes[1].set_xticks(range(len(vocab)))
axes[1].set_xticklabels(vocab, rotation=45, ha="right", fontsize=8)
axes[1].set_yticks([0, 1])
axes[1].set_yticklabels(["Sentence 1\n(Aureliano)", "Sentence 2\n(Macondo)"])
axes[1].set_title("BoW Vector Visualisation (2 sentences × 20-dim vocab)")
plt.colorbar(im, ax=axes[1], label="count")
plt.tight_layout()
plt.show()

# ── TF-IDF matrix ─────────────────────────────────────────────────────────────
corpus = [
    "Colonel Aureliano Buendia remembered the afternoon and the ice his father showed him",
    "Macondo was a village of adobe houses built on the bank of a clear river",
    "Melquiades the gypsy brought magnets and dragged iron objects from every house in Macondo",
]
vectorizer = TfidfVectorizer()
X          = vectorizer.fit_transform(corpus)
feat_names = vectorizer.get_feature_names_out()

fig2, ax2 = plt.subplots(figsize=(14, 3))
im2 = ax2.imshow(X.toarray(), aspect="auto", cmap="Blues")
ax2.set_xticks(range(len(feat_names)))
ax2.set_xticklabels(feat_names, rotation=45, ha="right", fontsize=8)
ax2.set_yticks(range(len(corpus)))
ax2.set_yticklabels(["Doc 1 — Aureliano", "Doc 2 — Macondo", "Doc 3 — Melquíades"])
ax2.set_title("TF-IDF Matrix — 3 sentences from Cien años de soledad")
plt.colorbar(im2, ax=ax2, label="TF-IDF weight")
plt.tight_layout()
plt.show()


---
## 2. Web Scraping

### 2a. Static page — `requests` + `BeautifulSoup`
Scraping book titles and ratings from **books.toscrape.com** — a site explicitly built for scraping practice.

In [ ]:
import requests
import time
import pandas as pd
from bs4 import BeautifulSoup

BASE_URL = "https://books.toscrape.com/catalogue/"
HEADERS  = {"User-Agent": "Mozilla/5.0 (NLP-Demo/1.0)"}

def scrape_books(n_pages: int = 2) -> pd.DataFrame:
    records = []
    for page in range(1, n_pages + 1):
        url = f"{BASE_URL}page-{page}.html"
        try:
            response = requests.get(url, headers=HEADERS, timeout=10)
            response.raise_for_status()
        except requests.RequestException as exc:
            print(f"[WARN] page {page} failed: {exc}")
            continue

        soup  = BeautifulSoup(response.text, "html.parser")
        books = soup.select("article.product_pod")
        for book in books:
            title  = book.h3.a["title"]
            rating = book.p["class"][1]          # 'One' … 'Five'
            price  = book.select_one(".price_color").text.strip()
            records.append({"title": title, "rating": rating, "price": price})

        print(f"  Page {page}: {len(books)} books scraped")
        time.sleep(1)   # politeness delay

    return pd.DataFrame(records)

df_books = scrape_books(n_pages=2)
print(f"\nTotal: {len(df_books)} books")
df_books.head(10)

In [ ]:
rating_order  = ["One", "Two", "Three", "Four", "Five"]
rating_counts = df_books["rating"].value_counts().reindex(rating_order, fill_value=0)

fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(rating_counts.index, rating_counts.values,
       color=["#d9534f","#f0ad4e","#f0ad4e","#5cb85c","#5cb85c"])
ax.set_xlabel("Star Rating"); ax.set_ylabel("Count")
ax.set_title("Rating Distribution — books.toscrape.com (40 books)")
plt.tight_layout(); plt.show()

### 2b. Dynamic page — `Selenium` (JavaScript-rendered content)

> **Requires**: `pip install selenium webdriver-manager` and Google Chrome installed.  
> Demonstrates the pattern needed when page content is injected by JavaScript after load.

In [ ]:
try:
    from selenium import webdriver
    from selenium.webdriver.common.by        import By
    from selenium.webdriver.support.ui       import WebDriverWait
    from selenium.webdriver.support          import expected_conditions as EC

    # ── Try Safari first (built-in on macOS, no extra driver needed) ──────────
    try:
        from selenium.webdriver.safari.options import Options as SafariOptions
        safari_opts = SafariOptions()
        driver = webdriver.Safari(options=safari_opts)
        _browser = "Safari"
    except Exception:
        # ── Fall back to Chrome if available ──────────────────────────────────
        from selenium.webdriver.chrome.service import Service
        from selenium.webdriver.chrome.options import Options
        from webdriver_manager.chrome          import ChromeDriverManager
        chrome_opts = Options()
        chrome_opts.add_argument("--headless")
        chrome_opts.add_argument("--no-sandbox")
        chrome_opts.add_argument("--disable-dev-shm-usage")
        driver   = webdriver.Chrome(service=Service(ChromeDriverManager().install()),
                                    options=chrome_opts)
        _browser = "Chrome"

    driver.get("https://books.toscrape.com/")
    WebDriverWait(driver, timeout=10).until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, "article.product_pod"))
    )

    selenium_books = []
    for el in driver.find_elements(By.CSS_SELECTOR, "article.product_pod")[:5]:
        title  = el.find_element(By.CSS_SELECTOR, "h3 > a").get_attribute("title")
        rating = el.find_element(By.CSS_SELECTOR, "p.star-rating").get_attribute("class").split()[-1]
        price  = el.find_element(By.CSS_SELECTOR, ".price_color").text
        selenium_books.append({"title": title, "rating": rating, "price": price})

    driver.quit()
    print(f"Selenium ({_browser}) — first 5 books:")
    for b in selenium_books:
        print(f"  {b['rating']:5s}  {b['price']}  {b['title'][:55]}")

except Exception as e:
    # ── No browser available: fall back to requests (same site, static HTML) ──
    print(f"[Selenium browser unavailable — falling back to requests]\n  ({e})\n")
    import requests
    from bs4 import BeautifulSoup
    resp = requests.get("https://books.toscrape.com/", timeout=10)
    soup = BeautifulSoup(resp.text, "html.parser")
    books = []
    for article in soup.select("article.product_pod")[:5]:
        title  = article.select_one("h3 > a")["title"]
        rating = article.select_one("p.star-rating")["class"][1]
        price  = article.select_one(".price_color").text.strip()
        books.append({"title": title, "rating": rating, "price": price})
    print("requests fallback — first 5 books:")
    for b in books:
        print(f"  {b['rating']:5s}  {b['price']}  {b['title'][:55]}")
    print("\n[Note] Selenium requires Chrome or Safari to demonstrate JS rendering.")
    print("       On macOS: enable Safari WebDriver via Safari → Develop → Allow Remote Automation")
    print("       Or install Chrome from https://www.google.com/chrome/")


---
## 3. Sentiment Analysis — Document, Sentence & Aspect Level

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def vader_label(compound: float) -> str:
    if compound >= 0.05:  return "POSITIVE"
    if compound <= -0.05: return "NEGATIVE"
    return "NEUTRAL"

# ── 3a: Document-level ────────────────────────────────────────────────────────
documents = [
    "This book is absolutely fantastic! I could not put it down.",
    "Terrible experience. The product broke on the first day.",
    "The conference room has a projector and chairs.",
    "I have mixed feelings — some chapters are great, others are boring.",
]

print("DOCUMENT-LEVEL SENTIMENT")
print("=" * 60)
for doc in documents:
    sc    = analyzer.polarity_scores(doc)
    label = vader_label(sc["compound"])
    print(f"  [{label:8s}]  compound={sc['compound']:+.3f}  │  {doc[:65]}")

In [ ]:
from nltk.tokenize import sent_tokenize

# ── 3b: Sentence-level ────────────────────────────────────────────────────────
review = (
    "The camera on this phone is absolutely stunning — photos come out crystal clear. "
    "However, the battery life is quite disappointing; it barely lasts a full day. "
    "The screen is decent, nothing special. "
    "Overall I am happy with the purchase despite the battery issue."
)

print("SENTENCE-LEVEL SENTIMENT")
print("=" * 60)
sentences = sent_tokenize(review)
sent_results = []
for i, sent in enumerate(sentences, 1):
    sc    = analyzer.polarity_scores(sent)
    label = vader_label(sc["compound"])
    sent_results.append((i, label, sc["compound"], sent))
    print(f"  S{i} [{label:8s}] compound={sc['compound']:+.3f}")
    print(f"      \"{sent}\"")

# Compound bar chart
colors    = {"POSITIVE": "#5cb85c", "NEGATIVE": "#d9534f", "NEUTRAL": "#f0ad4e"}
compounds = [r[2] for r in sent_results]
labels    = [f"S{r[0]}" for r in sent_results]
bar_cols  = [colors[r[1]] for r in sent_results]

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(labels, compounds, color=bar_cols)
ax.axhline(0,     color="black", linewidth=0.8)
ax.axhline( 0.05, color="green", linewidth=0.7, linestyle="--")
ax.axhline(-0.05, color="red",   linewidth=0.7, linestyle="--")
ax.set_ylabel("Compound score")
ax.set_title("Sentence-Level Sentiment")
patches = [mpatches.Patch(color=v, label=k) for k, v in colors.items()]
ax.legend(handles=patches, fontsize=8, loc="lower right")
plt.tight_layout(); plt.show()

In [ ]:
import spacy

try:
    nlp_spacy = spacy.load("en_core_web_sm")
except OSError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)
    nlp_spacy = spacy.load("en_core_web_sm")

# ── 3c: Aspect-level ─────────────────────────────────────────────────────────
ASPECTS = {
    "battery":  ["battery", "charge", "charging"],
    "camera":   ["camera", "photo", "picture", "lens"],
    "screen":   ["screen", "display", "resolution"],
    "price":    ["price", "cost", "expensive", "cheap", "value"],
    "build":    ["build", "quality", "material", "design"],
}

def aspect_sentiment(text: str) -> dict:
    aspect_results = {a: [] for a in ASPECTS}
    for sent in sent_tokenize(text):
        sc = analyzer.polarity_scores(sent)
        for aspect, keywords in ASPECTS.items():
            if any(kw in sent.lower() for kw in keywords):
                aspect_results[aspect].append((sc["compound"], sent))
    return aspect_results

phone_review = (
    "The camera takes stunning 4K video and the photos are amazing. "
    "Battery life is terrible — it dies after 4 hours of light use. "
    "The build quality feels premium, very solid in hand. "
    "For the price, I expected better battery performance. "
    "The screen resolution is crisp and the display is vibrant."
)

print("ASPECT-LEVEL SENTIMENT")
print("=" * 60)
asp_results = aspect_sentiment(phone_review)
summary = {}
for aspect, mentions in asp_results.items():
    if mentions:
        avg = sum(s for s, _ in mentions) / len(mentions)
        summary[aspect] = avg
        print(f"\n  ASPECT: {aspect.upper()} → [{vader_label(avg)}] avg={avg:+.3f}")
        for score, sent in mentions:
            print(f"    ({score:+.3f}) \"{sent}\"")

# Summary chart
if summary:
    fig, ax = plt.subplots(figsize=(7, 3))
    asp_colors = ["#5cb85c" if s > 0.05 else "#d9534f" if s < -0.05 else "#f0ad4e"
                  for s in summary.values()]
    ax.barh(list(summary.keys()), list(summary.values()), color=asp_colors)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Average compound score")
    ax.set_title("Aspect-Level Sentiment Summary")
    plt.tight_layout(); plt.show()

---
## 4. Three Approaches to Sentiment Analysis

| Approach | Method | Library |
|---|---|---|
| **Lexicon-based** | VADER score lookup | `vaderSentiment` |
| **Classical ML** | TF-IDF + Logistic Regression | `scikit-learn` |
| **Deep Learning** | Fine-tuned `DistilBERT` | `transformers` |

In [ ]:
# ── 4a: Lexicon-based (VADER) ─────────────────────────────────────────────────
test_sents = [
    "This is GREAT!!!",
    "Not bad at all.",
    "The food was good but the service was terrible.",
    "I don't think this is a good idea.",
    "Meh, it's okay I guess.",
    "I HATE waiting in line for hours!!",
    "Absolutely love the new design — very clean.",
]
print("LEXICON-BASED (VADER)")
print("=" * 70)
for s in test_sents:
    sc = analyzer.polarity_scores(s)
    lbl = vader_label(sc["compound"])
    print(f"  [{lbl:8s}]  pos={sc['pos']:.2f}  neu={sc['neu']:.2f}  neg={sc['neg']:.2f}  │  {s}")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model             import LogisticRegression
from sklearn.model_selection          import train_test_split
from sklearn.metrics                  import classification_report, ConfusionMatrixDisplay
from sklearn.pipeline                 import Pipeline

# ── 4b: Classical ML — TF-IDF + Logistic Regression ──────────────────────────
train_texts = [
    # positive
    "I love this product, it works perfectly",
    "Absolutely fantastic service, highly recommend",
    "Best purchase I have ever made, outstanding quality",
    "Great value for money, very happy with this",
    "Excellent performance, exceeded my expectations",
    "Wonderful experience from start to finish",
    "Super fast delivery and amazing quality",
    # negative
    "This is the worst product I have ever bought",
    "Terrible quality, broke after one day",
    "Complete waste of money, very disappointed",
    "Horrible service, never buying again",
    "Awful experience, do not recommend to anyone",
    "Poor quality, totally not worth it",
    "Disgusting customer support, very rude staff",
    # neutral
    "The package arrived on Tuesday",
    "It is a standard product with average features",
    "I received the item and it is as described",
    "The manual is included in the box",
    "Product dimensions are 30 by 20 centimetres",
    "I have not tried it yet",
    "The colour is blue as shown in the photo",
]
train_labels = ["positive"] * 7 + ["negative"] * 7 + ["neutral"] * 7

X_tr, X_te, y_tr, y_te = train_test_split(
    train_texts, train_labels, test_size=0.30, random_state=42, stratify=train_labels
)

ml_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1)),
    ("clf",   LogisticRegression(max_iter=500, random_state=42)),
])
ml_pipeline.fit(X_tr, y_tr)

print("CLASSICAL ML — TF-IDF + Logistic Regression")
print("=" * 55)
print(classification_report(y_te, ml_pipeline.predict(X_te), zero_division=0))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_estimator(ml_pipeline, X_te, y_te, ax=ax, cmap="Blues")
ax.set_title("Confusion Matrix (test set)")
plt.tight_layout(); plt.show()

new_examples = [
    "Absolutely love it!",
    "It arrived today in a box.",
    "Total garbage, broken on arrival.",
]
print("\nNew examples:")
for text, pred in zip(new_examples, ml_pipeline.predict(new_examples)):
    print(f"  [{pred:8s}]  {text}")

In [ ]:
# ── 4c: Deep Learning — HuggingFace DistilBERT (SST-2) ───────────────────────
try:
    from transformers import pipeline as hf_pipeline

    sa_model = hf_pipeline(
        "sentiment-analysis",
        model="distilbert-base-uncased-finetuned-sst-2-english",
        truncation=True,
    )
    deep_examples = [
        "I absolutely love this! Best thing I bought all year.",
        "Not bad at all, actually better than I expected.",
        "Terrible. It stopped working after two days.",
        "The item is a standard grey box with no documentation.",
        "I don't think this is terrible, but it could be better.",
        "This EXCEEDED every single expectation I had.",
    ]
    print("DEEP LEARNING — DistilBERT (SST-2 fine-tuned)")
    print("=" * 60)
    for text, res in zip(deep_examples, sa_model(deep_examples)):
        print(f"  [{res['label']:8s}]  score={res['score']:.4f}  │  {text[:65]}")
except Exception as exc:
    print(f"[transformers unavailable]: {exc}")
    print("Install: pip install transformers torch")

In [ ]:
# ── 4d: Comparison — VADER vs Classical ML ───────────────────────────────────
comparison_texts = [
    "I love this product, it is absolutely amazing!",
    "Not bad, quite decent actually.",
    "Terrible service, very disappointed.",
    "The item arrived on Wednesday in a brown box.",
    "I don't think it is a bad choice.",
]
ml_map  = {"positive": 1.0, "neutral": 0.0, "negative": -1.0}
lex_sc  = [analyzer.polarity_scores(t)["compound"] for t in comparison_texts]
ml_sc   = [ml_map[p] for p in ml_pipeline.predict(comparison_texts)]

x, w = np.arange(len(comparison_texts)), 0.3
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - w / 2, lex_sc, w, label="VADER (lexicon)",  color="steelblue")
ax.bar(x + w / 2, ml_sc,  w, label="LogReg (ML)",      color="darkorange")
ax.axhline(0, color="black", linewidth=0.7)
ax.set_xticks(x)
ax.set_xticklabels([f"T{i+1}" for i in range(len(comparison_texts))])
ax.set_title("Lexicon vs Classical ML — 5 sentences")
ax.legend()
plt.tight_layout(); plt.show()

print("\nSentences:")
for i, t in enumerate(comparison_texts, 1):
    print(f"  T{i}: {t}")

---
## 5. Retrieval-Augmented Generation (RAG)

**Architecture:**

```
Query ──► Embedder ──► Vector DB (FAISS)
                            │
                       Top-k chunks
                            │
              Prompt = context + question
                            │
                        LLM response
```

> **Libraries used**: `sentence-transformers` (embeddings), `faiss-cpu` (vector search).  
> The LLM call is mocked here; swap `mock_llm()` for any real endpoint (OpenAI, Ollama, etc.).

In [ ]:
# ── 5a: Document corpus (NLP knowledge base) ─────────────────────────────────
documents = [
    "Tokenization is the process of splitting text into individual words or subwords called tokens.",
    "Stop words are common words such as 'the', 'is', 'at' that carry little semantic meaning.",
    "Lemmatization reduces words to their base form; 'running' becomes 'run'.",
    "Stemming is a cruder form of normalization that chops off word suffixes.",
    "TF-IDF stands for Term Frequency–Inverse Document Frequency and measures word importance.",
    "A Bag of Words model represents text as a vector of token counts, ignoring word order.",
    "Word embeddings like Word2Vec map words to dense continuous vectors in a semantic space.",
    "BERT is a transformer model pre-trained with masked language modelling on large corpora.",
    "Sentiment analysis classifies text as positive, negative, or neutral.",
    "Named Entity Recognition (NER) identifies and classifies entities like persons, organizations, and locations.",
    "Dependency parsing reveals grammatical relationships between words in a sentence.",
    "A language model predicts the probability of the next token given a preceding context.",
    "RAG — Retrieval-Augmented Generation — combines a retriever and a generative LLM.",
    "FAISS is a library by Meta AI for efficient similarity search over dense vector embeddings.",
    "Cosine similarity measures the angle between two vectors; 1.0 means identical direction.",
]

print(f"Corpus: {len(documents)} NLP-fact documents")
for i, d in enumerate(documents):
    print(f"  [{i:02d}] {d[:80]}")

In [ ]:
# ── 5b: Encode documents and build FAISS index ───────────────────────────────
try:
    from sentence_transformers import SentenceTransformer
    import faiss

    print("Loading sentence-transformers model …")
    encoder = SentenceTransformer("all-MiniLM-L6-v2")   # 22 MB — fast & accurate

    print("Encoding corpus …")
    doc_embeddings = encoder.encode(documents, convert_to_numpy=True, normalize_embeddings=True)
    print(f"Embeddings shape: {doc_embeddings.shape}")   # (15, 384)

    # Build a flat inner-product index (cosine sim ≡ inner product when normalised)
    dim   = doc_embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(doc_embeddings)
    print(f"FAISS index ready — {index.ntotal} vectors")
    _rag_available = True

except Exception as exc:
    print(f"[sentence-transformers / faiss unavailable]: {exc}")
    print("Install: pip install sentence-transformers faiss-cpu")
    _rag_available = False

In [ ]:
# ── 5c: Retrieval function ────────────────────────────────────────────────────
def retrieve(query: str, top_k: int = 3) -> list[tuple[float, str]]:
    """Return (score, document) pairs for the top-k most relevant chunks."""
    q_emb = encoder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores, indices = index.search(q_emb, top_k)
    return [(float(scores[0][i]), documents[indices[0][i]]) for i in range(top_k)]


demo_queries = [
    "What is TF-IDF?",
    "How does RAG work?",
    "What is the difference between stemming and lemmatization?",
]

if _rag_available:
    for q in demo_queries:
        print(f"\nQuery: \"{q}\"")
        for rank, (score, doc) in enumerate(retrieve(q), 1):
            print(f"  #{rank}  sim={score:.4f}  {doc[:80]}")

In [ ]:
# ── 5d: Mock LLM + full RAG pipeline ─────────────────────────────────────────
def mock_llm(prompt: str) -> str:
    """
    Stub that echoes the retrieved context as if it were a model answer.
    Replace the body with an actual API call (OpenAI, Ollama, etc.).
    """
    lines = [l.strip() for l in prompt.split("\n") if l.strip().startswith("-")]
    context = " ".join(l.lstrip("- ") for l in lines[:2])
    return f"[Mock answer] Based on the retrieved context: {context[:160]} …"

def rag_answer(query: str, top_k: int = 3) -> str:
    chunks = retrieve(query, top_k=top_k)
    context_block = "\n".join(f"- {doc}" for _, doc in chunks)
    prompt = (
        f"You are a helpful NLP assistant. Answer the question using only the context below.\n\n"
        f"Context:\n{context_block}\n\n"
        f"Question: {query}\nAnswer:"
    )
    return mock_llm(prompt)

# ── Run demo ──────────────────────────────────────────────────────────────────
if _rag_available:
    rag_questions = [
        "What is a Bag of Words?",
        "How does retrieval-augmented generation work?",
        "What does BERT stand for?",
    ]
    for q in rag_questions:
        print(f"\n{'─'*60}")
        print(f"Q: {q}")
        print(f"A: {rag_answer(q)}")
else:
    print("[RAG demo skipped — libraries unavailable]")

In [ ]:
# ── 5e: Cosine similarity heatmap — query vs corpus ───────────────────────────
if _rag_available:
    vis_queries = [
        "What is TF-IDF?",
        "How does RAG work?",
        "What is the difference between stemming and lemmatization?",
    ]
    q_embs  = encoder.encode(vis_queries, convert_to_numpy=True, normalize_embeddings=True)
    # cosine sim = inner product (vectors are already normalised)
    sim_mat = q_embs @ doc_embeddings.T          # (3, 15)

    fig, ax = plt.subplots(figsize=(14, 3.5))
    im = ax.imshow(sim_mat, aspect="auto", cmap="coolwarm", vmin=0, vmax=1)
    ax.set_yticks(range(len(vis_queries)))
    ax.set_yticklabels([f"Q{i+1}" for i in range(len(vis_queries))])
    ax.set_xticks(range(len(documents)))
    ax.set_xticklabels([f"D{i:02d}" for i in range(len(documents))], fontsize=7)
    ax.set_title("Cosine Similarity — Queries × Corpus")
    plt.colorbar(im, ax=ax, label="cosine similarity")

    # Annotate with score
    for r in range(len(vis_queries)):
        for c in range(len(documents)):
            ax.text(c, r, f"{sim_mat[r, c]:.2f}", ha="center", va="center",
                    fontsize=6, color="black")

    plt.tight_layout(); plt.show()

    print("\nQuery legend:")
    for i, q in enumerate(vis_queries, 1):
        print(f"  Q{i}: {q}")
    print("\nTop-3 retrieval per query:")
    for i, q in enumerate(vis_queries, 1):
        top3 = sorted(enumerate(sim_mat[i-1]), key=lambda x: x[1], reverse=True)[:3]
        hits = ", ".join(f"D{idx:02d}({sc:.2f})" for idx, sc in top3)
        print(f"  Q{i}: {hits}")

---
## 6. Word Embeddings & Semantic Analogies

Static word representations like BoW and TF-IDF treat every word as an isolated symbol — they can not capture that *"king"* and *"queen"* are related, or that *"Paris"* is to *"France"* what *"Berlin"* is to *"Germany"*.

**Word embeddings** (Word2Vec, GloVe, FastText) map each word to a **dense vector** $\mathbf{v} \in \mathbb{R}^d$ so that:
- semantically similar words are **close** in vector space
- relational analogies survive as **vector arithmetic**:
$$\mathbf{v}_{\text{king}} - \mathbf{v}_{\text{man}} + \mathbf{v}_{\text{woman}} \approx \mathbf{v}_{\text{queen}}$$

### 6a. Conceptual 2-D Visualisation

Recreating the slide diagram: four words in a hand-crafted 2-D space where *gender* runs along the x-axis and *royalty* along the y-axis.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── 6a: Hand-crafted 2-D embedding space (like the slide) ────────────────────
# Each word is a point in [gender (x), royalty (y)] space
words_2d = {
    "king":    (0.5,  2.2),
    "man":     (0.7,  0.8),
    "queen":   (2.4,  2.3),
    "woman":   (2.6,  0.9),
    "royalty": (1.5,  1.55),
    "boy":     (0.9,  0.35),
    "girl":    (2.3,  0.35),
    "prince":  (0.6,  1.7),
    "princess":(2.3,  1.8),
}

fig, ax = plt.subplots(figsize=(7, 6))
ax.set_xlabel("← masculine / feminine →", fontsize=11)
ax.set_ylabel("← commoner / royalty →",   fontsize=11)
ax.set_title("Word Embedding 2-D Projection\n(Gender × Royalty axes)", fontsize=12)
ax.set_xlim(-0.3, 3.2); ax.set_ylim(-0.3, 3.2)

# Colour-code by gender
colours = {
    "king": "blue", "man": "blue", "boy": "blue", "prince": "blue",
    "queen": "red",  "woman": "red",  "girl": "red",  "princess": "red",
    "royalty": "green",
}

for word, (x, y) in words_2d.items():
    c = colours.get(word, "grey")
    ax.scatter(x, y, s=90, color=c, zorder=3)
    ax.annotate(word, (x, y), xytext=(6, 4), textcoords="offset points", fontsize=10)

# Dashed analogies: king–man and queen–woman (gender axis)
for (a, b) in [("king", "queen"), ("man", "woman"), ("prince", "princess"), ("boy", "girl")]:
    ax.annotate("", xy=words_2d[b], xytext=words_2d[a],
                arrowprops=dict(arrowstyle="<->", color="grey", lw=1.2, linestyle="dotted"))

# Highlight the analogy king − man + woman ≈ queen
kx, ky = words_2d["king"]
mx, my = words_2d["man"]
wx, wy = words_2d["woman"]
pred = np.array([kx - mx + wx, ky - my + wy])
ax.scatter(*pred, s=180, marker="*", color="orange", zorder=5, label="king−man+woman (predicted)")
ax.annotate("predicted\n'queen'?", pred, xytext=(10, -25), textcoords="offset points",
            fontsize=9, color="darkorange",
            arrowprops=dict(arrowstyle="->", color="darkorange"))

patches = [
    mpatches.Patch(color="blue",  label="masculine"),
    mpatches.Patch(color="red",   label="feminine"),
    mpatches.Patch(color="green", label="abstract"),
    mpatches.Patch(color="orange",label="vector arithmetic prediction"),
]
ax.legend(handles=patches, fontsize=9, loc="upper left")
plt.tight_layout(); plt.show()
print("The orange star is very close to 'queen' — the analogy holds in embedding space!")


### 6b. Pre-trained Word Embeddings with Gensim

`gensim` ships with a model downloader for common pre-trained embeddings.  
Here we load **GloVe 50-d** (~66 MB, downloaded once and cached) and explore real word relationships.

> **Install**: `pip install gensim`  
> First run downloads the model — subsequent runs are instant.


In [ ]:
try:
    import gensim.downloader as gensim_api

    print("Loading GloVe 50-d vectors (downloads ~66 MB on first run) …")
    glove = gensim_api.load("glove-wiki-gigaword-50")
    print(f"Loaded {len(glove):,} words, each a 50-dimensional vector\n")
    _glove_available = True

    # ── Word similarity ──────────────────────────────────────────────────────
    pairs = [
        ("king",  "queen"),
        ("man",   "woman"),
        ("paris", "london"),
        ("cat",   "dog"),
        ("cat",   "airplane"),
    ]
    print("WORD SIMILARITY (cosine)")
    print("─" * 40)
    for w1, w2 in pairs:
        sim = glove.similarity(w1, w2)
        print(f"  {w1:10s} — {w2:10s}  →  {sim:.4f}")

    # ── Most similar words ───────────────────────────────────────────────────
    print("\nMOST SIMILAR WORDS")
    print("─" * 40)
    for word in ["python", "nlp", "paris", "music"]:
        if word in glove:
            top5 = glove.most_similar(word, topn=5)
            print(f"  {word}: {', '.join(w for w, _ in top5)}")

    # ── Analogy: king − man + woman ≈ queen ──────────────────────────────────
    print("\nANALOGY:  king − man + woman = ?")
    result = glove.most_similar(positive=["king", "woman"], negative=["man"], topn=5)
    for word, score in result:
        print(f"  {word:15s}  score={score:.4f}")

    print("\nANALOGY:  paris − france + italy = ?")
    result2 = glove.most_similar(positive=["paris", "italy"], negative=["france"], topn=5)
    for word, score in result2:
        print(f"  {word:15s}  score={score:.4f}")

except Exception as exc:
    _glove_available = False
    print(f"[gensim unavailable or download failed]: {exc}")
    print("Install: pip install gensim   (model downloads automatically)")


In [ ]:
from sklearn.decomposition import PCA

# ── 6c: PCA 2-D visualisation of GloVe word clusters ─────────────────────────
if _glove_available:
    word_groups = {
        "royalty":   ["king", "queen", "prince", "princess", "throne", "crown"],
        "countries": ["france", "germany", "italy", "spain", "japan", "china"],
        "capitals":  ["paris", "berlin", "rome", "madrid", "tokyo", "beijing"],
        "animals":   ["cat", "dog", "horse", "lion", "tiger", "elephant"],
        "sports":    ["football", "basketball", "tennis", "swimming", "running"],
    }
    all_words  = [w for grp in word_groups.values() for w in grp]
    all_vecs   = np.array([glove[w] for w in all_words])

    # Reduce to 2-D with PCA
    pca     = PCA(n_components=2)
    coords  = pca.fit_transform(all_vecs)
    var     = pca.explained_variance_ratio_

    group_colours = {
        "royalty":   "gold",
        "countries": "steelblue",
        "capitals":  "royalblue",
        "animals":   "forestgreen",
        "sports":    "tomato",
    }

    fig, ax = plt.subplots(figsize=(10, 7))
    i = 0
    for group, words in word_groups.items():
        n   = len(words)
        pts = coords[i:i+n]
        ax.scatter(pts[:, 0], pts[:, 1], s=80, label=group, color=group_colours[group])
        for j, word in enumerate(words):
            ax.annotate(word, pts[j], fontsize=8, xytext=(4, 3), textcoords="offset points")
        i += n

    ax.set_xlabel(f"PC1 ({var[0]*100:.1f}% variance)", fontsize=11)
    ax.set_ylabel(f"PC2 ({var[1]*100:.1f}% variance)", fontsize=11)
    ax.set_title("GloVe 50-d Word Vectors — PCA 2-D Projection\n"
                 "Semantically related words cluster together", fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

    # ── Country → Capital relationship ────────────────────────────────────────
    print("VECTOR ARITHMETIC: Country → Capital")
    print("─" * 45)
    pairs_cc = [("france","paris"), ("germany","berlin"), ("italy","rome")]
    offsets  = [glove[cap] - glove[cty] for cty, cap in pairs_cc]
    avg_offset = np.mean(offsets, axis=0)
    for test_country in ["spain", "japan", "china"]:
        predicted_vec = glove[test_country] + avg_offset
        # find closest word to predicted vector
        top = glove.similar_by_vector(predicted_vec, topn=3)
        print(f"  {test_country:10s} + offset → {[w for w, _ in top]}")
else:
    print("[Skipped — gensim / GloVe not available]")


---
## 7. Transformer Self-Attention

> *"Attention is all you need"* — Vaswani et al., 2017

Before Transformers, RNNs processed tokens **sequentially** — they couldn't parallelize and struggled with long-range dependencies.  
Self-attention lets **every token simultaneously look at every other token** and decide how much to attend to it.

### The Scaled Dot-Product Attention Formula

For a sequence of tokens with input matrix $X \in \mathbb{R}^{n \times d}$:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

where $Q = XW^Q$, $K = XW^K$, $V = XW^V$ are learned projections (Query, Key, Value).

- The **softmax** produces a probability distribution over all tokens for each query
- $\sqrt{d_k}$ prevents dot-products from growing too large and saturating softmax
- The output for each token is a **weighted sum of all token values** — context-aware!

### 7a. From-Scratch NumPy Implementation


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

np.random.seed(42)

# ── Scaled dot-product attention — pure NumPy ─────────────────────────────────
def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    e = np.exp(x - x.max(axis=axis, keepdims=True))   # numerical stability
    return e / e.sum(axis=axis, keepdims=True)

def scaled_dot_product_attention(Q: np.ndarray,
                                  K: np.ndarray,
                                  V: np.ndarray,
                                  mask: np.ndarray | None = None
                                  ) -> tuple[np.ndarray, np.ndarray]:
    """
    Q, K, V : (seq_len, d_k)
    Returns contextual output and attention weights matrix.
    """
    d_k     = Q.shape[-1]
    scores  = (Q @ K.T) / np.sqrt(d_k)         # (seq_len, seq_len)
    if mask is not None:
        scores = scores + mask                  # add -∞ where masked
    weights = softmax(scores, axis=-1)           # probability rows
    output  = weights @ V                       # (seq_len, d_k)
    return output, weights

# ── Example: sentence "The bank by the river" ─────────────────────────────────
tokens  = ["The", "bank", "by", "the", "river"]
n, d_k  = len(tokens), 8       # 5 tokens, 8-d key/query space

# Random projections — in a real transformer these are learned
W_Q = np.random.randn(d_k, d_k) * 0.3
W_K = np.random.randn(d_k, d_k) * 0.3
W_V = np.random.randn(d_k, d_k) * 0.3

# Dummy input: each token is a one-hot + positional encoding
def positional_encoding(pos: int, d: int) -> np.ndarray:
    enc = np.zeros(d)
    for i in range(d):
        angle = pos / (10000 ** (2 * (i // 2) / d))
        enc[i] = np.sin(angle) if i % 2 == 0 else np.cos(angle)
    return enc

X = np.stack([positional_encoding(i, d_k) for i in range(n)])   # (5, 8)

Q_mat = X @ W_Q
K_mat = X @ W_K
V_mat = X @ W_V

output, attn_weights = scaled_dot_product_attention(Q_mat, K_mat, V_mat)

print("Input shape  :", X.shape)
print("Q/K/V shape  :", Q_mat.shape)
print("Output shape :", output.shape)
print("\nAttention weight matrix (each row sums to 1.0):")
print(f"{'':8s}", "  ".join(f"{t:6s}" for t in tokens))
for i, row in enumerate(attn_weights):
    print(f"{tokens[i]:8s}", "  ".join(f"{v:6.3f}" for v in row))


In [ ]:
# ── 7b: Attention heatmap ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ─ Left: our random-weight attention ─
im1 = axes[0].imshow(attn_weights, cmap="Blues", vmin=0, vmax=1)
axes[0].set_xticks(range(n)); axes[0].set_yticks(range(n))
axes[0].set_xticklabels(tokens, fontsize=10)
axes[0].set_yticklabels(tokens, fontsize=10)
axes[0].set_xlabel("Key (attended to)", fontsize=10)
axes[0].set_ylabel("Query (attending from)", fontsize=10)
axes[0].set_title("Self-Attention Weights\n(random init — no fine-tuning)", fontsize=10)
for i in range(n):
    for j in range(n):
        axes[0].text(j, i, f"{attn_weights[i,j]:.2f}",
                     ha="center", va="center", fontsize=8,
                     color="white" if attn_weights[i, j] > 0.5 else "black")
plt.colorbar(im1, ax=axes[0])

# ─ Right: simulated "bank"→"river" attention (matching the slide) ─
slide_attn = np.array([
    [0.25, 0.20, 0.20, 0.20, 0.15],   # "The"
    [0.41, 0.15, 0.09, 0.13, 0.72],   # "bank" strongly attends to "river"
    [0.15, 0.20, 0.30, 0.20, 0.15],   # "by"
    [0.20, 0.10, 0.20, 0.30, 0.20],   # "the"
    [0.15, 0.60, 0.10, 0.10, 0.05],   # "river" attends to "bank"
])
# renormalise rows so they sum to 1
slide_attn /= slide_attn.sum(axis=1, keepdims=True)

im2 = axes[1].imshow(slide_attn, cmap="Reds", vmin=0, vmax=0.75)
axes[1].set_xticks(range(n)); axes[1].set_yticks(range(n))
axes[1].set_xticklabels(tokens, fontsize=10)
axes[1].set_yticklabels(tokens, fontsize=10)
axes[1].set_xlabel("Key (attended to)", fontsize=10)
axes[1].set_ylabel("Query (attending from)", fontsize=10)
axes[1].set_title('"bank" → "river" (high attention)\nSemantic context selects correct sense', fontsize=10)
for i in range(n):
    for j in range(n):
        axes[1].text(j, i, f"{slide_attn[i,j]:.2f}",
                     ha="center", va="center", fontsize=8,
                     color="white" if slide_attn[i, j] > 0.5 else "black")
plt.colorbar(im2, ax=axes[1])
plt.suptitle("Self-Attention Heatmaps — 'The bank by the river'", fontsize=12, y=1.01)
plt.tight_layout(); plt.show()

print('"bank" attends most to "river" (0.72) → context resolves ambiguity: river bank, not financial bank')


In [ ]:
# ── 7c: Causal (masked) attention — GPT-style ────────────────────────────────
# Causal mask: token i cannot attend to tokens j > i (future positions)
causal_mask = np.full((n, n), -1e9)
for i in range(n):
    for j in range(i + 1):
        causal_mask[i, j] = 0.0    # allow attending to past and self

_, causal_weights = scaled_dot_product_attention(Q_mat, K_mat, V_mat, mask=causal_mask)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

im3 = axes[0].imshow(attn_weights, cmap="Blues", vmin=0, vmax=1)
axes[0].set_xticks(range(n)); axes[0].set_yticks(range(n))
axes[0].set_xticklabels(tokens, fontsize=10); axes[0].set_yticklabels(tokens, fontsize=10)
axes[0].set_title("BERT-style (Bidirectional)\nAll tokens attend to all tokens", fontsize=10)
plt.colorbar(im3, ax=axes[0])

im4 = axes[1].imshow(causal_weights, cmap="Reds", vmin=0, vmax=1)
axes[1].set_xticks(range(n)); axes[1].set_yticks(range(n))
axes[1].set_xticklabels(tokens, fontsize=10); axes[1].set_yticklabels(tokens, fontsize=10)
axes[1].set_title("GPT-style (Causal / Autoregressive)\nToken can only attend to past tokens", fontsize=10)
# Draw the upper-triangle mask visually
for i in range(n):
    for j in range(n):
        v = causal_weights[i, j]
        axes[1].text(j, i, f"{v:.2f}" if v > 0.001 else "✗",
                     ha="center", va="center", fontsize=8,
                     color="white" if v > 0.5 else ("grey" if v < 0.001 else "black"))
plt.colorbar(im4, ax=axes[1])
plt.suptitle("Bidirectional vs Causal Attention", fontsize=12)
plt.tight_layout(); plt.show()

print("\nKey difference:")
print("  BERT  — sees the full sentence → great for understanding/classification")
print("  GPT   — sees only the past     → great for generation (no data leakage)")
print("  T5    — encoder is bidirectional, decoder is causal + cross-attention")


In [ ]:
# ── 7d: BERT attention weights via HuggingFace (optional) ─────────────────────
try:
    import torch
    from transformers import BertTokenizer, BertModel

    print("Loading BERT (bert-base-uncased) …")
    tokenizer_bert = BertTokenizer.from_pretrained("bert-base-uncased")
    model_bert     = BertModel.from_pretrained("bert-base-uncased",
                                               output_attentions=True)
    model_bert.eval()

    sentence   = "The bank by the river flooded the road"
    inputs     = tokenizer_bert(sentence, return_tensors="pt")
    with torch.no_grad():
        outputs = model_bert(**inputs)

    # outputs.attentions: tuple of (1, 12 heads, seq_len, seq_len) for each layer
    tok_labels = tokenizer_bert.convert_ids_to_tokens(inputs["input_ids"][0])
    # Use layer 6, average over heads
    layer_attn = outputs.attentions[5][0]          # (12, seq, seq)
    avg_attn   = layer_attn.mean(dim=0).numpy()    # (seq, seq)

    fig, ax = plt.subplots(figsize=(9, 7))
    im = ax.imshow(avg_attn, cmap="YlOrRd")
    ax.set_xticks(range(len(tok_labels))); ax.set_yticks(range(len(tok_labels)))
    ax.set_xticklabels(tok_labels, rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(tok_labels, fontsize=9)
    ax.set_title(f"BERT Real Attention (layer 6, avg 12 heads)\n\"{sentence}\"", fontsize=10)
    plt.colorbar(im, ax=ax, label="attention weight")
    plt.tight_layout(); plt.show()

    print("\nHigh-attention pairs (> 0.15):")
    for i in range(len(tok_labels)):
        for j in range(len(tok_labels)):
            if i != j and avg_attn[i, j] > 0.15:
                print(f"  {tok_labels[i]:12s} → {tok_labels[j]:12s}  {avg_attn[i,j]:.3f}")

except Exception as exc:
    print(f"[BERT attention demo unavailable]: {exc}")
    print("Install: pip install transformers torch")


---
## 8. Transformer Families: BERT, GPT, T5

The three dominant transformer architectures each use a different **pre-training objective** and excel at different tasks:

| Model | Architecture | Pre-training Objective | Best for |
|-------|-------------|------------------------|----------|
| **BERT** | Encoder-only | **MLM** — predict masked tokens | Classification, NER, Q&A |
| **GPT** | Decoder-only | **CLM** — predict next token | Text generation, chatbots |
| **T5** | Encoder–Decoder | **Span corruption** — reconstruct masked spans | Translation, summarisation |

### Attention Pattern Comparison

| | Token sees future? | Parallel encoding? | Seq-to-seq? |
|---|---|---|---|
| BERT | ✗ (bidirectional) | ✓ | ✗ |
| GPT | ✓ (causal) | ✓ | ✗ |
| T5 | Enc: ✗ / Dec: ✓ | ✓ | ✓ |

### 8a. Pre-training Objectives Illustrated


In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── 8a: Pre-training objective visualisations ────────────────────────────────
sentence = ["The", "bank", "by", "the", "river", "flooded"]

fig, axes = plt.subplots(1, 3, figsize=(15, 3))

def draw_token_row(ax, tokens, colours, labels=None, title="", subtitle=""):
    n = len(tokens)
    for i, (tok, c) in enumerate(zip(tokens, colours)):
        rect = plt.Rectangle((i, 0), 0.9, 0.7, color=c, linewidth=1.5, edgecolor="black")
        ax.add_patch(rect)
        ax.text(i + 0.45, 0.35, tok, ha="center", va="center",
                fontsize=10, fontweight="bold",
                color="white" if c in ("#e74c3c","#2c3e50","#7f8c8d") else "black")
        if labels and labels[i]:
            ax.text(i + 0.45, -0.25, labels[i], ha="center", va="center",
                    fontsize=8, color="#e74c3c", fontstyle="italic")
    ax.set_xlim(-0.1, n + 0.2); ax.set_ylim(-0.5, 1.2)
    ax.axis("off")
    ax.set_title(title, fontsize=11, fontweight="bold", pad=8)
    ax.text(n / 2, -0.42, subtitle, ha="center", fontsize=9, color="#555")

# BERT — Masked Language Model (MLM)
bert_cols  = ["#3498db", "#e74c3c", "#3498db", "#3498db", "#3498db", "#3498db"]
bert_toks  = ["The", "[MASK]", "by", "the", "river", "flooded"]
bert_lbls  = [None, "predict: 'bank'", None, None, None, None]
draw_token_row(axes[0], bert_toks, bert_cols, bert_lbls,
               title="BERT — MLM",
               subtitle="Predict randomly masked tokens (15% of input)")

# GPT — Causal Language Model (CLM)
gpt_cols   = ["#27ae60"] * 5 + ["#e74c3c"]
gpt_toks   = ["The", "bank", "by", "the", "river", "→ ?"]
gpt_lbls   = [None, None, None, None, None, "predict next"]
draw_token_row(axes[1], gpt_toks, gpt_cols, gpt_lbls,
               title="GPT — CLM",
               subtitle="Predict next token given all previous tokens")

# T5 — Span Corruption
t5_cols    = ["#3498db", "#7f8c8d", "#3498db", "#3498db", "#3498db", "#3498db"]
t5_toks    = ["The", "<X>", "by", "the", "river", "flooded"]
t5_lbls    = [None, "reconstruct: 'bank'", None, None, None, None]
draw_token_row(axes[2], t5_toks, t5_cols, t5_lbls,
               title="T5 — Span Corruption",
               subtitle="Mask contiguous spans, reconstruct in decoder")

plt.suptitle("Transformer Pre-training Objectives", fontsize=13, y=1.04)
plt.tight_layout(); plt.show()


In [ ]:
# ── 8b: Using BERT, GPT-2, and T5 from HuggingFace ───────────────────────────
try:
    from transformers import (
        BertTokenizer, BertForMaskedLM,
        GPT2Tokenizer, GPT2LMHeadModel,
        T5Tokenizer, T5ForConditionalGeneration,
    )
    import torch

    # ─ BERT: Fill-mask ────────────────────────────────────────────────────────
    print("=" * 60)
    print("BERT — Fill-Mask (MLM)")
    print("=" * 60)
    bert_tok   = BertTokenizer.from_pretrained("bert-base-uncased")
    bert_mlm   = BertForMaskedLM.from_pretrained("bert-base-uncased")
    bert_mlm.eval()

    masked_sents = [
        "The bank by the [MASK] flooded.",
        "The [MASK] approved the loan request.",
        "She studied natural language [MASK].",
    ]
    for sent in masked_sents:
        inputs   = bert_tok(sent, return_tensors="pt")
        mask_idx = (inputs["input_ids"] == bert_tok.mask_token_id).nonzero(as_tuple=True)[1][0]
        with torch.no_grad():
            logits = bert_mlm(**inputs).logits
        top5_ids    = logits[0, mask_idx].topk(5).indices
        top5_tokens = [bert_tok.decode([tid]).strip() for tid in top5_ids]
        display = sent.replace("[MASK]", f"[{', '.join(top5_tokens[:3])}]")
        print(f"\n  Input : {sent}")
        print(f"  Top-5 : {top5_tokens}")

    # ─ GPT-2: Text generation ─────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("GPT-2 — Text Generation (CLM)")
    print("=" * 60)
    gpt2_tok = GPT2Tokenizer.from_pretrained("gpt2")
    gpt2_lm  = GPT2LMHeadModel.from_pretrained("gpt2")
    gpt2_lm.eval()
    gpt2_tok.pad_token = gpt2_tok.eos_token

    prompts = [
        "Natural language processing is",
        "The main challenge of sentiment analysis is",
    ]
    for prompt in prompts:
        ids = gpt2_tok(prompt, return_tensors="pt").input_ids
        with torch.no_grad():
            out = gpt2_lm.generate(ids, max_new_tokens=30, do_sample=True,
                                   temperature=0.8, top_p=0.9,
                                   pad_token_id=gpt2_tok.eos_token_id)
        gen_text = gpt2_tok.decode(out[0], skip_special_tokens=True)
        print(f"\n  Prompt : \"{prompt}\"")
        print(f"  GPT-2  : \"{gen_text}\"")

    # ─ T5: Translation / summarisation ────────────────────────────────────────
    print("\n" + "=" * 60)
    print("T5 — Seq-to-Seq (Translation)")
    print("=" * 60)
    t5_tok = T5Tokenizer.from_pretrained("t5-small")
    t5_mod = T5ForConditionalGeneration.from_pretrained("t5-small")
    t5_mod.eval()

    tasks = [
        "translate English to French: Natural language processing makes machines understand text.",
        "summarize: Natural language processing (NLP) is a branch of artificial intelligence "
        "that focuses on the interaction between computers and humans through natural language. "
        "The goal is to enable computers to understand, interpret, and generate human language.",
    ]
    for task_input in tasks:
        ids = t5_tok(task_input, return_tensors="pt", max_length=512, truncation=True).input_ids
        with torch.no_grad():
            out = t5_mod.generate(ids, max_new_tokens=60)
        result = t5_tok.decode(out[0], skip_special_tokens=True)
        task_label = task_input.split(":")[0].upper()
        print(f"\n  Task   : {task_label}")
        print(f"  Output : \"{result}\"")

except Exception as exc:
    print(f"[HuggingFace demo unavailable]: {exc}")
    print("Install: pip install transformers torch")


---
## 9. Sentence Embeddings & Semantic Search

Word2Vec and GloVe give **one vector per word**, ignoring context.  
BERT gives context-aware token representations, but they're not optimised for **sentence-level** semantic similarity.

**Sentence-BERT (SBERT)** fine-tunes a siamese BERT network using a contrastive loss on sentence pairs, so that:
- Semantically equivalent sentences land **close together**
- Unrelated sentences are **far apart**
- Semantic search reduces to a fast **nearest-neighbour** lookup

### Why SBERT over raw BERT?

| Method | "I like dogs" vs "I love cats" | Computation |
|--------|-------------------------------|-------------|
| TF-IDF cosine | low (different vocabulary) | fast |
| BERT cross-encoder | accurate but slow | $O(n^2)$ |
| **SBERT** | high (semantically similar) | $O(n)$ + index |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ── 9a: TF-IDF vs SBERT similarity comparison ────────────────────────────────
sentence_pairs = [
    # Semantically equivalent, different surface form
    ("The car drove down the road.",              "An automobile travelled along the highway."),
    ("I enjoy playing football on weekends.",     "Soccer is my favourite weekend activity."),
    ("The student passed the examination.",       "She succeeded in her test."),
    # Similar topic, different meaning
    ("The bank approved the loan.",               "The river bank was flooded."),
    # Completely unrelated
    ("NLP stands for natural language processing.", "The pizza arrived cold and soggy."),
]

print("SIMILARITY COMPARISON: TF-IDF vs SBERT")
print("=" * 72)

all_sents = [s for pair in sentence_pairs for s in pair]

# TF-IDF
tfidf_vec  = TfidfVectorizer().fit(all_sents)
tfidf_sims = []
for s1, s2 in sentence_pairs:
    v1, v2 = tfidf_vec.transform([s1]), tfidf_vec.transform([s2])
    tfidf_sims.append(float(cosine_similarity(v1, v2)[0][0]))

# SBERT
try:
    from sentence_transformers import SentenceTransformer
    sbert = SentenceTransformer("all-MiniLM-L6-v2")
    sbert_sims = []
    for s1, s2 in sentence_pairs:
        e1 = sbert.encode([s1], normalize_embeddings=True)
        e2 = sbert.encode([s2], normalize_embeddings=True)
        sbert_sims.append(float(e1 @ e2.T))
    _sbert_available = True
except Exception:
    sbert_sims = [None] * len(sentence_pairs)
    _sbert_available = False

for i, (s1, s2) in enumerate(sentence_pairs):
    tf = tfidf_sims[i]
    sb = sbert_sims[i]
    sb_str = f"{sb:.3f}" if sb is not None else "N/A"
    print(f"\nPair {i+1}:")
    print(f"  S1: \"{s1[:60]}\"")
    print(f"  S2: \"{s2[:60]}\"")
    print(f"  TF-IDF: {tf:.3f}  │  SBERT: {sb_str}")


In [ ]:
# ── 9b: Side-by-side bar chart comparison ────────────────────────────────────
if _sbert_available:
    labels  = [f"P{i+1}" for i in range(len(sentence_pairs))]
    x       = np.arange(len(labels))
    w       = 0.35

    fig, ax = plt.subplots(figsize=(10, 4))
    bars1 = ax.bar(x - w/2, tfidf_sims, w, label="TF-IDF cosine", color="steelblue")
    bars2 = ax.bar(x + w/2, sbert_sims, w, label="SBERT cosine",  color="tomato")

    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=11)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("Cosine similarity", fontsize=11)
    ax.set_title("TF-IDF vs SBERT Semantic Similarity\n"
                 "(P1–P3: paraphrases, P4: polysemy, P5: unrelated)", fontsize=11)
    ax.legend()
    ax.axhline(0.5, color="grey", linestyle="--", linewidth=0.8, label="0.5 threshold")
    plt.tight_layout(); plt.show()

    print("\nInsight:")
    print("  P1-P3: SBERT correctly identifies paraphrases; TF-IDF sees different words → low score")
    print("  P4:    Both models rate 'bank' sentences lower — SBERT still higher due to context")
    print("  P5:    Both correctly rate unrelated sentences near 0")


In [ ]:
from sklearn.decomposition import PCA

# ── 9c: PCA clustering of sentence embeddings ────────────────────────────────
if _sbert_available:
    sentence_clusters = {
        "Weather / Climate": [
            "It rains a lot in London in autumn.",
            "The weather in the UK is often cloudy.",
            "Heavy snowfall is expected this winter.",
            "Temperatures dropped below zero last night.",
        ],
        "Finance / Banking": [
            "The stock market crashed after the announcement.",
            "Interest rates were raised by the central bank.",
            "Inflation is at a 40-year high.",
            "Investors are worried about the economic outlook.",
        ],
        "Technology / AI": [
            "Large language models are trained on massive corpora.",
            "GPT-4 can generate human-like text.",
            "Natural language processing enables machines to understand text.",
            "BERT uses bidirectional self-attention.",
        ],
        "Sports": [
            "The team scored a last-minute goal.",
            "She won the gold medal at the Olympics.",
            "The tennis player served 20 aces in the match.",
            "Basketball requires speed, agility, and teamwork.",
        ],
    }

    all_sents_cls  = [s for grp in sentence_clusters.values() for s in grp]
    all_labels_cls = [lbl for lbl, sents in sentence_clusters.items() for _ in sents]
    embeddings_cls = sbert.encode(all_sents_cls, normalize_embeddings=True)

    # PCA to 2-D
    pca2  = PCA(n_components=2)
    pts   = pca2.fit_transform(embeddings_cls)
    var2  = pca2.explained_variance_ratio_

    colours_map = {
        "Weather / Climate": "royalblue",
        "Finance / Banking": "tomato",
        "Technology / AI":   "forestgreen",
        "Sports":            "darkorchid",
    }

    fig, ax = plt.subplots(figsize=(10, 7))
    for group, colour in colours_map.items():
        mask = [i for i, l in enumerate(all_labels_cls) if l == group]
        ax.scatter(pts[mask, 0], pts[mask, 1], s=120,
                   color=colour, label=group, zorder=3)
        for idx in mask:
            ax.annotate(all_sents_cls[idx][:35] + "…",
                        pts[idx], fontsize=6.5, xytext=(4, 3),
                        textcoords="offset points", color=colour)

    ax.set_xlabel(f"PC1 ({var2[0]*100:.1f}% variance)", fontsize=11)
    ax.set_ylabel(f"PC2 ({var2[1]*100:.1f}% variance)", fontsize=11)
    ax.set_title("SBERT Sentence Embeddings — PCA 2-D Projection\n"
                 "Sentences in the same topic cluster together", fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

    # Semantic search demo
    query   = "How does the economy affect interest rates?"
    q_emb   = sbert.encode([query], normalize_embeddings=True)
    sims    = (q_emb @ embeddings_cls.T)[0]
    top_idx = sims.argsort()[::-1][:5]

    print(f"\nSemantic Search Query: \"{query}\"")
    print("Top-5 results:")
    for rank, idx in enumerate(top_idx, 1):
        print(f"  #{rank}  [{all_labels_cls[idx]:20s}]  sim={sims[idx]:.4f}  "
              f"\"{all_sents_cls[idx]}\"")
else:
    print("[Skipped — sentence-transformers not available]")


---
## 10. Current Challenges in NLP

Despite remarkable progress, NLP systems still face serious open problems:

| Challenge | Description | Mitigation |
|-----------|-------------|------------|
| **Hallucination** | LLMs generate plausible but false facts | RAG, grounding, fact-checking |
| **Bias** | Stereotypes from training data surface in outputs | Debiasing, diverse corpora, RLHF |
| **Low-resource languages** | ~7,000 languages; most research is English-centric | Multilingual models (mBERT, XLM-R) |
| **Interpretability** | Black-box decisions are hard to explain | SHAP, LIME, attention maps |
| **Computational cost** | GPT-4 ≈ $200M to train | LoRA, PEFT, quantisation |
| **Privacy** | Text contains PII — GDPR compliance required | Differential privacy, data anonymisation |

### 10a. Hallucination Demo

We simulate hallucination by asking a language model a factual question and checking its output against a known ground truth.


In [ ]:
# ── 10a: Hallucination — GPT-2 generation on factual questions ───────────────
# GPT-2 (117M params, 2019) has no web access and often "confabulates" facts.
# This demo illustrates why grounding (RAG) is necessary for factual tasks.
try:
    from transformers import pipeline as hf_pipeline

    gen = hf_pipeline("text-generation", model="gpt2",
                      max_new_tokens=60, do_sample=True, temperature=0.9,
                      top_p=0.9, pad_token_id=50256)

    factual_questions = [
        "The capital of Australia is",
        "The inventor of the telephone was",
        "The boiling point of water at sea level is",
    ]
    ground_truth = [
        "Canberra",
        "Alexander Graham Bell",
        "100°C (212°F)",
    ]

    print("HALLUCINATION DEMO — GPT-2 factual questions")
    print("=" * 65)
    for question, truth in zip(factual_questions, ground_truth):
        outputs   = gen(question, num_return_sequences=3, truncation=True)
        responses = [o["generated_text"][len(question):].split(".")[0].strip()
                     for o in outputs]
        print(f"\n  Q : \"{question}\"")
        print(f"  GT: {truth}")
        for i, resp in enumerate(responses, 1):
            flag = "✓" if truth.lower().split()[0] in resp.lower() else "✗ (possible hallucination)"
            print(f"  R{i}: \"{resp[:80]}\"  {flag}")

    print("\n\nNote: GPT-2 sometimes gets facts right by chance, but often confabulates.")
    print("Newer models (GPT-4, Claude) are better, but still hallucinate.")
    print("→ RAG (Section 5) grounds answers in a verified document store.")

except Exception as exc:
    print(f"[GPT-2 unavailable]: {exc}")


### 10b. Bias in NLP Models

Language models learn from human-generated text, which contains societal biases.  
The classic test: **word association** — words like *"doctor"*, *"nurse"*, *"engineer"*, *"secretary"* often retrieve gendered associations in pre-trained models.

We measure this using **cosine similarity** between occupation embeddings and gender word embeddings.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── 10b: Gender bias in GloVe embeddings ─────────────────────────────────────
# We use the "gender direction": v_he - v_she as a proxy axis

if _glove_available:
    gender_direction = glove["he"] - glove["she"]   # positive = male-associated
    gender_direction /= np.linalg.norm(gender_direction)

    occupations = [
        "doctor", "nurse", "engineer", "secretary",
        "programmer", "receptionist", "pilot", "teacher",
        "scientist", "librarian", "ceo", "assistant",
        "lawyer", "babysitter", "surgeon", "cleaner",
    ]

    # Filter words present in GloVe vocab
    occupations = [w for w in occupations if w in glove]
    biases = {w: float(glove[w] @ gender_direction) for w in occupations}
    sorted_items = sorted(biases.items(), key=lambda kv: kv[1])

    words_sorted  = [k for k, _ in sorted_items]
    scores_sorted = [v for _, v in sorted_items]
    colours_sorted = ["tomato" if s < 0 else "steelblue" for s in scores_sorted]

    fig, ax = plt.subplots(figsize=(9, 6))
    bars = ax.barh(words_sorted, scores_sorted, color=colours_sorted)
    ax.axvline(0, color="black", linewidth=1.2)
    ax.set_xlabel("← female-associated          male-associated →", fontsize=10)
    ax.set_title("Gender Bias in GloVe 50-d Embeddings\n"
                 "(projection onto 'he'–'she' direction)", fontsize=11)
    ax.set_xlim(-0.65, 0.65)
    for bar, score in zip(bars, scores_sorted):
        ax.text(score + (0.01 if score >= 0 else -0.01), bar.get_y() + bar.get_height()/2,
                f"{score:+.3f}", va="center", ha="left" if score >= 0 else "right", fontsize=8)
    plt.tight_layout(); plt.show()

    print("\nObservation:")
    print("  Occupations like 'nurse', 'secretary', 'babysitter' cluster on the feminine side")
    print("  while 'engineer', 'programmer', 'surgeon' lean masculine — reflecting training data bias.")
    print("\nMitigation strategies:")
    print("  - Hard debiasing (project out the gender subspace from occupation vectors)")
    print("  - Data augmentation (counterfactual data generation)")
    print("  - Post-training alignment (RLHF with human feedback on fairness)")
else:
    # Demonstrate with SBERT if GloVe not available
    if _sbert_available:
        print("BIAS DEMO with SBERT (GloVe not available)")
        male_words   = ["he", "man", "him", "his", "male", "boy"]
        female_words = ["she", "woman", "her", "hers", "female", "girl"]
        occupations_demo = ["doctor", "nurse", "engineer", "secretary", "programmer", "receptionist"]

        m_emb = sbert.encode(male_words,   normalize_embeddings=True).mean(axis=0)
        f_emb = sbert.encode(female_words, normalize_embeddings=True).mean(axis=0)
        o_emb = sbert.encode(occupations_demo, normalize_embeddings=True)

        for word, vec in zip(occupations_demo, o_emb):
            m_sim = float(vec @ m_emb)
            f_sim = float(vec @ f_emb)
            lean  = "male" if m_sim > f_sim else "female"
            print(f"  {word:15s}  male={m_sim:.3f}  female={f_sim:.3f}  → leans {lean}")
    else:
        print("[Bias demo skipped — neither GloVe nor SBERT available]")


### 10c. Interpretability — Explaining Model Decisions

Black-box models are hard to trust in production. Three common explanation techniques:

- **Attention visualisation**: show which tokens the model focused on (Section 7 heatmaps)
- **LIME** (Local Interpretable Model-agnostic Explanations): perturb input, fit local linear model
- **SHAP** (SHapley Additive exPlanations): assign each token a contribution score based on game theory

Below we implement a lightweight **LIME-style perturbation** for our TF-IDF + LogReg classifier from Section 4.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.linear_model import Ridge

# ── 10c: LIME-style perturbation explanation ──────────────────────────────────
def lime_explain(pipeline, text: str, n_samples: int = 300,
                 target_class: str = "positive") -> dict:
    """
    Lightweight LIME implementation for a sklearn text pipeline.
    Returns a dict {word: importance_score} for the target_class.
    """
    words   = text.split()
    n_words = len(words)

    np.random.seed(0)
    # Binary presence matrix: each row is a perturbed sample
    presence       = np.random.randint(0, 2, size=(n_samples, n_words))
    presence[0, :] = 1   # first sample is the original text

    # Build perturbed texts
    perturbed_texts = []
    for row in presence:
        masked = " ".join(w if row[i] else "" for i, w in enumerate(words)).strip()
        perturbed_texts.append(masked if masked else "empty")

    # Get probabilities from the classifier
    probas = pipeline.predict_proba(perturbed_texts)
    class_idx = list(pipeline.classes_).index(target_class)
    y         = probas[:, class_idx]

    # Distance kernel: closer to original = higher weight
    distances = np.sqrt(np.sum((presence - 1) ** 2, axis=1))
    kernel_w  = np.exp(-distances ** 2 / (2 * (0.75 ** 2)))

    # Fit weighted local linear model
    ridge   = Ridge(alpha=1.0)
    ridge.fit(presence, y, sample_weight=kernel_w)
    importance = dict(zip(words, ridge.coef_))
    return importance

def plot_lime(importance: dict, text: str, prediction: str, title: str = ""):
    sorted_imp = sorted(importance.items(), key=lambda kv: abs(kv[1]), reverse=True)[:12]
    words_s    = [k for k, _ in sorted_imp]
    scores_s   = [v for _, v in sorted_imp]
    colours_s  = ["forestgreen" if s > 0 else "tomato" for s in scores_s]

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.barh(words_s[::-1], scores_s[::-1], color=colours_s[::-1])
    ax.axvline(0, color="black", linewidth=1)
    ax.set_xlabel("LIME importance (+ → positive, − → negative)", fontsize=10)
    ax.set_title(f"{title}\nPrediction: {prediction.upper()}", fontsize=11)
    patches = [mpatches.Patch(color="forestgreen", label="pushes towards positive"),
               mpatches.Patch(color="tomato",      label="pushes towards negative")]
    ax.legend(handles=patches, fontsize=9)
    plt.tight_layout(); plt.show()

# Examples to explain
explain_examples = [
    ("Absolutely love this! Best purchase I have ever made.",     "positive"),
    ("Complete waste of money, broken and terrible.",             "negative"),
    ("The product arrived in a standard brown box on Tuesday.",   "neutral"),
    ("Not bad at all, actually better than I expected somehow.",  "positive"),
]

print("LIME-STYLE EXPLANATIONS — TF-IDF + LogReg (from Section 4)")
print("=" * 60)
for text, expected in explain_examples:
    pred_label = ml_pipeline.predict([text])[0]
    importance = lime_explain(ml_pipeline, text, n_samples=500, target_class="positive")
    match = "✓" if pred_label == expected else "✗"
    print(f"\nText     : \"{text[:70]}\"")
    print(f"Expected : {expected:8s}  Predicted : {pred_label:8s}  {match}")
    top3 = sorted(importance.items(), key=lambda kv: kv[1], reverse=True)[:3]
    bot3 = sorted(importance.items(), key=lambda kv: kv[1])[:3]
    print(f"Top +3   : {top3}")
    print(f"Top −3   : {bot3}")
    plot_lime(importance, text, pred_label,
              title=f"LIME: \"{text[:55]}…\"" if len(text) > 55 else f"LIME: \"{text}\"")


---
## Summary

This notebook covered the complete NLP pipeline from the course slides:

| Section | Topics | Key Libraries |
|---------|--------|---------------|
| 1 | NLP Pipeline: tokenise → stopwords → lemmatise → BoW + TF-IDF | `nltk`, `sklearn` |
| 2 | Web Scraping: static (requests/BS4) and dynamic (Selenium) | `requests`, `bs4`, `selenium` |
| 3 | Sentiment Analysis: document, sentence, and aspect level | `vaderSentiment`, `spacy` |
| 4 | Three Approaches: Lexicon, Classical ML, Deep Learning | `sklearn`, `transformers` |
| 5 | RAG: vector DB (FAISS) + retrieval + mock LLM | `sentence-transformers`, `faiss` |
| 6 | Word Embeddings: 2-D analogy plots, GloVe similarity, PCA clusters | `gensim` |
| 7 | Transformer Self-Attention: QKV from scratch, heatmaps, BERT attention | `numpy`, `transformers` |
| 8 | Transformer Families: BERT (MLM), GPT (CLM), T5 (span corruption) | `transformers` |
| 9 | Sentence Embeddings: SBERT vs TF-IDF, PCA clustering, semantic search | `sentence-transformers` |
| 10 | Current Challenges: hallucination, bias, LIME interpretability | `transformers`, `gensim`, `sklearn` |

### Next Steps (Workshop)
Build a complete pipeline:  
`scrape reviews` → `preprocess` → `train sentiment classifier` → `deploy as Streamlit app`
